# 02 — Graph Construction (CholecT50)

**Goal:** Build causal knowledge graphs from the parsed surgical action triplets.
Each graph models the relationships between instruments and anatomical targets,
connected by surgical verbs (actions).

**Videos:** VID01, VID05, VID40

## Graph Schema Design

### Node Types

We have two categories of nodes, forming a **bipartite-like** graph:

| Node Type | Source | Examples | Attributes |
|-----------|--------|----------|------------|
| **Instrument** | `instrument` column | grasper, hook, bipolar, clipper, scissors, irrigator | `node_type='instrument'`, `id` |
| **Target** | `target` column | gallbladder, cystic_plate, cystic_duct, liver, omentum, ... | `node_type='target'`, `id` |

### Edge Definition

Edges connect an **instrument node** to a **target node**. The edge label is
the **verb** (surgical action). Since the same instrument–target pair can be
connected by different verbs, we use a **MultiDiGraph** (directed, allows
parallel edges with different keys).

```
  [grasper] --retract--> [gallbladder]
  [grasper] --grasp----> [gallbladder]
  [hook]    --dissect--> [cystic_plate]
```

### Edge Attributes

Each edge stores temporal and frequency information:

| Attribute | Type | Description |
|-----------|------|-------------|
| `verb` | str | The surgical action (edge label / key) |
| `triplet_id` | int | CholecT50 triplet class ID |
| `triplet_label` | str | Full triplet string (e.g. `grasper,retract,gallbladder`) |
| `frequency` | int | Number of frames where this triplet appears |
| `frame_list` | list[int] | Ordered list of frame IDs where this triplet is active |
| `first_frame` | int | First frame of occurrence |
| `last_frame` | int | Last frame of occurrence |
| `duration_frames` | int | Span from first to last frame |
| `timestamp_start` | float | Start time in seconds |
| `timestamp_end` | float | End time in seconds |

### Why MultiDiGraph?

- **Directed:** Instrument → Target (the instrument acts *on* the target)
- **Multi:** Same instrument–target pair can have multiple verb edges
  (e.g., grasper→gallbladder with *retract* AND *grasp*)
- **Temporal attributes:** Allow downstream causal / sequential analysis

### Graph-Level Metadata

Each graph object carries metadata as graph attributes:
- `video`: Video ID (e.g. VID01)
- `total_frames`: Number of annotated frames
- `total_triplet_rows`: Total triplet annotations
- `unique_triplets`: Number of distinct triplet classes observed

### Summary Diagram

```
                    GRAPH SCHEMA
                    ============

  ┌──────────────┐                    ┌──────────────┐
  │  INSTRUMENT   │                    │    TARGET     │
  │  (node_type=  │    verb (edge)     │  (node_type=  │
  │  'instrument')│ ═══════════════>   │  'target')    │
  │              │   frequency=N      │              │
  │  e.g. grasper│   frame_list=[..]  │  e.g. liver  │
  └──────────────┘   first/last_frame  └──────────────┘

  NetworkX: nx.MultiDiGraph
  - Nodes: instruments + targets
  - Edges: one per unique (instrument, verb, target) triplet
  - Edge key: verb name
  - Edge attrs: frequency, frame_list, timestamps
```

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import networkx as nx
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'
GRAPH_DIR = PROJECT_ROOT / 'outputs' / 'graphs'
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

print('Parsed triplet CSVs:', sorted(os.listdir(TRIPLET_DIR)))

Parsed triplet CSVs: ['1_triplets.csv', '40_triplets.csv', '5_triplets.csv', 'VID01_triplets.csv', 'VID05_triplets.csv', 'VID40_triplets.csv', 'all_triplets.csv']


## Quick Data Recap

Load one video to verify the schema maps to our parsed data.

In [2]:
df = pd.read_csv(TRIPLET_DIR / 'VID01_triplets.csv')
print(f'VID01: {len(df)} rows, {df["frame"].nunique()} frames')
print(f'Columns: {list(df.columns)}')
print(f'\nUnique instruments: {sorted(df["instrument"].unique())}')
print(f'Unique targets:     {sorted(df["target"].unique())}')
print(f'Unique verbs:       {sorted(df["verb"].unique())}')
print(f'Unique triplets:    {df["triplet_id"].nunique()}')
df.head()

VID01: 2560 rows, 1543 frames
Columns: ['video', 'frame', 'timestamp_sec', 'entry_index', 'is_multi_label', 'triplet_id', 'triplet_label', 'instrument_id', 'instrument', 'verb_id', 'verb', 'target_id', 'target', 'is_valid', 'is_null', 'num_triplets_in_frame']

Unique instruments: ['bipolar', 'clipper', 'grasper', 'hook', 'irrigator', 'scissors']
Unique targets:     ['abdominal_wall_cavity', 'blood_vessel', 'cystic_artery', 'cystic_duct', 'cystic_pedicle', 'fluid', 'gallbladder', 'liver', 'null_target', 'specimen_bag']
Unique verbs:       ['aspirate', 'clip', 'coagulate', 'cut', 'dissect', 'grasp', 'irrigate', 'null_verb', 'pack', 'retract']
Unique triplets:    22


,video,frame,timestamp_sec,entry_index,is_multi_label,triplet_id,triplet_label,instrument_id,instrument,verb_id,verb,target_id,target,is_valid,is_null,num_triplets_in_frame
0,VID01,0,0.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
1,VID01,1,1.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
2,VID01,2,2.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
3,VID01,3,3.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1
4,VID01,4,4.0,0,False,7,"grasper,grasp,gallbladder",0,grasper,0,grasp,0,gallbladder,True,False,1


## Schema Validation: Edge Aggregation Preview

Before building the actual graph, preview how triplets aggregate into edges.
Each unique `(instrument, verb, target)` becomes one edge with a frequency count.

In [3]:
# Filter out null_verb entries for graph construction
df_valid = df[~df['is_null']].copy()
print(f'Valid (non-null) triplets: {len(df_valid)} / {len(df)}')

# Preview edge aggregation
edge_preview = (
    df_valid.groupby(['instrument', 'verb', 'target'])
    .agg(
        frequency=('frame', 'count'),
        first_frame=('frame', 'min'),
        last_frame=('frame', 'max'),
    )
    .sort_values('frequency', ascending=False)
    .reset_index()
)

print(f'\nUnique edges (instrument→verb→target): {len(edge_preview)}')
print(f'\nTop 10 edges by frequency:')
display(edge_preview.head(10))

Valid (non-null) triplets: 2247 / 2560

Unique edges (instrument→verb→target): 19

Top 10 edges by frequency:


,instrument,verb,target,frequency,first_frame,last_frame
0,grasper,grasp,gallbladder,711,0,1433
1,hook,dissect,gallbladder,524,493,1430
2,grasper,retract,liver,359,1164,1640
3,grasper,grasp,specimen_bag,125,1478,1717
4,irrigator,aspirate,fluid,117,291,1624
5,grasper,retract,gallbladder,113,524,1203
6,bipolar,coagulate,blood_vessel,100,325,466
7,clipper,clip,cystic_duct,78,698,843
8,grasper,pack,gallbladder,27,1512,1540
9,hook,dissect,cystic_duct,21,251,271


In [4]:
# Node count preview
instruments = df_valid['instrument'].unique()
targets = df_valid['target'].unique()

print(f'Expected graph nodes: {len(instruments)} instruments + {len(targets)} targets = {len(instruments) + len(targets)} total')
print(f'Expected graph edges: {len(edge_preview)} unique (instrument, verb, target) combinations')
print(f'\nGraph density estimate: {len(edge_preview) / (len(instruments) * len(targets)):.2f}')

Expected graph nodes: 6 instruments + 9 targets = 15 total
Expected graph edges: 19 unique (instrument, verb, target) combinations

Graph density estimate: 0.35


---

**Schema is validated.** Next step: implement `src/graph_builder.py` to construct
the actual `nx.MultiDiGraph` from parsed CSVs.